In [9]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

orders = pd.read_csv("suya_orders.csv")
orders

# Make a copy so the original data is not changed
data = orders.copy()
data

# Check missing values and duplicates
print(data.isnull().sum())
print("Duplicates:", data.duplicated().sum())

# Clean the copy
data["Date"] = pd.to_datetime(data["Date"])
data["Customer_Age"] = data["Customer_Age"].fillna(
    data["Customer_Age"].median()
)
data["Payment_Method"] = data["Payment_Method"].fillna("Unknown")
data = data.drop_duplicates()

# Create useful columns
data["Month"] = data["Date"].dt.month_name()
data["Day"] = data["Date"].dt.day_name()


Order_ID            0
Date                0
Branch              0
Item                0
Quantity            0
Unit_Price_NGN      0
Total_NGN           0
Payment_Method    153
Customer_Age      281
Rating            396
Is_Delivery         0
dtype: int64
Duplicates: 50


In [10]:
branch_sales = (
    data.groupby("Branch")["Total_NGN"]
    .sum()
    .sort_values(ascending=False)
)

print(branch_sales)


Branch
Lagos Yaba           5177700
Abuja Wuse           3981050
Kaduna Central       3224700
Kano Sabon Gari      2858000
Port Harcourt GRA    2003200
Ibadan Bodija        1863900
Jos Terminus         1749200
Enugu Ogui           1620900
Name: Total_NGN, dtype: int64


In [13]:
item_summary = data.groupby("Item").agg(
    Number_of_Orders=("Item", "count"),
    Portions_Sold=("Quantity", "sum"),
    Total_Sales=("Total_NGN", "sum")
).sort_values(
    "Number_of_Orders",
    ascending=False
)

display(item_summary)


,Number_of_Orders,Portions_Sold,Total_Sales
Item,,,
Beef Suya,1495,3603,5651950
Chicken Suya,1142,3294,6297850
Kilishi,708,1724,3640650
Ram Suya,627,1721,4542700
Gizzard,599,1486,1886400
Soft Drink,379,862,459100


In [19]:
orders["Date"] = pd.to_datetime(orders["Date"])

orders["Month_Number"] = orders["Date"].dt.month
orders["Month"] = orders["Date"].dt.month_name()


In [20]:
print(orders[["Date", "Month_Number", "Month"]].head())


        Date  Month_Number      Month
0 2025-08-03             8     August
1 2025-03-14             3      March
2 2025-11-11            11   November
3 2025-03-09             3      March
4 2025-09-07             9  September


In [21]:
monthly_sales = (
    orders.groupby(["Month_Number", "Month"])["Total_NGN"]
    .sum()
    .sort_index()
)

print(monthly_sales)


Month_Number  Month    
1             January      1716450
2             February     1585200
3             March        1719350
4             April        1923250
5             May          1979500
6             June         2040400
7             July         2152850
8             August       2043500
9             September    1651050
10            October      1839800
11            November     2087400
12            December     1952050
Name: Total_NGN, dtype: int64


In [22]:
payment_counts = data["Payment_Method"].value_counts()
print(payment_counts)


Payment_Method
Cash        1915
Transfer    1696
POS         1187
Unknown      152
Name: count, dtype: int64


In [23]:
delivery_ratings = (
    data.groupby("Is_Delivery")["Rating"]
    .mean()
)

print(delivery_ratings)


Is_Delivery
No     4.146304
Yes    3.771028
Name: Rating, dtype: float64


In [24]:
print("Average age:", data["Customer_Age"].mean())

age_bins = [0, 17, 25, 35, 45, 100]
age_labels = [
    "Under 18",
    "18–25",
    "26–35",
    "36–45",
    "46+"
]

data["Age_Group"] = pd.cut(
    data["Customer_Age"],
    bins=age_bins,
    labels=age_labels
)

print(data["Age_Group"].value_counts())


Average age: 29.254949494949496
Age_Group
26–35       2453
18–25       1142
36–45        918
Under 18     354
46+           83
Name: count, dtype: int64


In [25]:
busy_days = data["Day"].value_counts()
print(busy_days)


Day
Wednesday    748
Saturday     740
Tuesday      737
Friday       710
Monday       682
Sunday       669
Thursday     664
Name: count, dtype: int64


In [26]:
quantity_rating_correlation = (
    data["Quantity"].corr(data["Rating"])
)

print(quantity_rating_correlation)


-0.014589439057958523


In [27]:
branch_ratings = (
    data.groupby("Branch")["Rating"]
    .mean()
    .sort_values(ascending=False)
)

print(branch_ratings)


Branch
Ibadan Bodija        4.081340
Abuja Wuse           4.065160
Jos Terminus         4.059140
Enugu Ogui           4.055703
Lagos Yaba           4.055156
Kano Sabon Gari      4.035200
Port Harcourt GRA    4.017316
Kaduna Central       3.976323
Name: Rating, dtype: float64


In [28]:
print("Sales by item:")
print(
    data.groupby("Item")["Total_NGN"]
    .sum()
    .sort_values(ascending=False)
)

print("\nRatings by delivery status:")
print(
    data.groupby("Is_Delivery")["Rating"]
    .mean()
)

print("\nSales by branch:")
print(branch_sales)

print("\nSales by month:")
print(monthly_sales)


Sales by item:
Item
Chicken Suya    6297850
Beef Suya       5651950
Ram Suya        4542700
Kilishi         3640650
Gizzard         1886400
Soft Drink       459100
Name: Total_NGN, dtype: int64

Ratings by delivery status:
Is_Delivery
No     4.146304
Yes    3.771028
Name: Rating, dtype: float64

Sales by branch:
Branch
Lagos Yaba           5177700
Abuja Wuse           3981050
Kaduna Central       3224700
Kano Sabon Gari      2858000
Port Harcourt GRA    2003200
Ibadan Bodija        1863900
Jos Terminus         1749200
Enugu Ogui           1620900
Name: Total_NGN, dtype: int64

Sales by month:
Month_Number  Month    
1             January      1716450
2             February     1585200
3             March        1719350
4             April        1923250
5             May          1979500
6             June         2040400
7             July         2152850
8             August       2043500
9             September    1651050
10            October      1839800
11            November    

Lagos Yaba is the highest-earning branch, Chicken Suya is the highest-revenue item, and July is the strongest sales month. Delivery orders receive lower ratings than non-delivery orders, while order quantity has almost no relationship with customer rating. To make more money, the business should promote Chicken Suya, improve delivery service, and copy the successful practices used by the Lagos Yaba branch and during July.